<a href="https://colab.research.google.com/github/lucajcc/VC-and-Sustainability/blob/main/Green_Patents_Fixed_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook processes patent data to identify 'green patents' based on CPC codes. It extracts relevant patent information, including filing dates and assignee/inventor locations, and then generates several output files:

1.  **`green_patents_raw.csv`**: A raw list of identified green patents with their CPC categories, filing dates, and associated US inventor and assignee states.
2.  **`green_patents_panel_inventor.csv`**: A panel dataset summarizing the number of unique green patents by inventor state, year, and CPC category.
3.  **`green_patents_panel_assignee.csv`**: A panel dataset summarizing the number of unique green patents by assignee state, year, and CPC category.

The process involves:

*   Mounting Google Drive to access data files.
*   Defining file paths and CPC codes for filtering.
*   Loading and filtering CPC data to identify green patents.
*   Loading application data to filter green patents by filing year.
*   Building a location lookup to map `location_id` to US states.
*   Loading assignee data and mapping assignees to US states.
*   Loading inventor data and mapping inventors to US states.
*   Aggregating data and saving the final CSV outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After mounting, you can load your data (e.g., a CSV file) like this:

```python
import pandas as pd
# Replace 'your_folder/your_file.csv' with your actual file path
# df = pd.read_csv('/content/drive/MyDrive/your_folder/your_file.csv')
# display(df.head())
```

In [ ]:
# Cell 2: SET YOUR FILE PATHS
# Change 'patent_data' to the name of your folder in Google Drive.
# If your files are directly in My Drive (no subfolder), use:
#   '/content/drive/MyDrive/g_cpc_current.tsv'  etc.

FOLDER = '/content/drive/MyDrive/SBLY210/patent_data'

CPC_FILE      = f'{FOLDER}/g_cpc_current.tsv'
APP_FILE      = f'{FOLDER}/g_application.tsv'
LOCATION_FILE = f'{FOLDER}/g_location_disambiguated.tsv'
INVENTOR_FILE = f'{FOLDER}/g_inventor_disambiguated.tsv'
ASSIGNEE_FILE = f'{FOLDER}/g_assignee_disambiguated.tsv'

YEAR_START = 2018
YEAR_END   = 2025
CPC_CODES  = ['Y02B', 'Y02C', 'Y02D', 'Y02E', 'Y02P', 'Y02T', 'Y02W']

US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY','DC'
}

import os, pandas as pd
all_ok = True
for label, path in [('CPC',      CPC_FILE),
                     ('Application', APP_FILE),
                     ('Location', LOCATION_FILE),
                     ('Inventor', INVENTOR_FILE),
                     ('Assignee', ASSIGNEE_FILE)]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e9
        print(f'  OK  {label:12s}: {path}  ({size:.2f} GB)')
    else:
        print(f'  NOT FOUND  {label}: {path}')
        print('             --> Update FOLDER above and re-run this cell.')
        all_ok = False
if all_ok:
    print('\nAll 5 files found. Ready to proceed.')

  OK  CPC         : /content/drive/MyDrive/SBLY210/patent_data/g_cpc_current.tsv  (3.38 GB)
  OK  Application : /content/drive/MyDrive/SBLY210/patent_data/g_application.tsv  (0.44 GB)
  OK  Location    : /content/drive/MyDrive/SBLY210/patent_data/g_location_disambiguated.tsv  (0.01 GB)
  OK  Inventor    : /content/drive/MyDrive/SBLY210/patent_data/g_inventor_disambiguated.tsv  (2.30 GB)
  OK  Assignee    : /content/drive/MyDrive/SBLY210/patent_data/g_assignee_disambiguated.tsv  (1.11 GB)

All 5 files found. Ready to proceed.


In [ ]:
# Cell 3: Load CPC file — extract green patent IDs
print('Loading CPC file...')
sample = pd.read_csv(CPC_FILE, sep='\t', nrows=3)
cols = list(sample.columns)
print('  Columns:', cols)

pid_col = next((c for c in cols if 'patent' in c.lower() and 'id' in c.lower()), cols[0])
cpc_col = next((c for c in cols if 'subclass' in c.lower() or
                                    'group'    in c.lower() or
                                    'cpc'      in c.lower()), cols[1])

# Corrected: Explicitly set cpc_col to 'cpc_subclass'
cpc_col = 'cpc_subclass'

print(f'  patent_id col: "{pid_col}"  |  cpc col: "{cpc_col}"')

cpc_chunks, scanned = [], 0
for i, chunk in enumerate(pd.read_csv(CPC_FILE, sep='\t', chunksize=500_000,
                                        usecols=[pid_col, cpc_col], low_memory=False)):
    chunk['cpc_top'] = chunk[cpc_col].astype(str).str[:4]
    match = chunk[chunk['cpc_top'].isin(CPC_CODES)][[pid_col, 'cpc_top']].copy()
    cpc_chunks.append(match)
    scanned += len(chunk)
    print(f'  Chunk {i+1}: {scanned:,} rows scanned | {sum(len(c) for c in cpc_chunks):,} green rows kept')

df_cpc = pd.concat(cpc_chunks, ignore_index=True)
df_cpc.columns = ['patent_id', 'cpc_category']
df_cpc['patent_id'] = df_cpc['patent_id'].astype(str)
green_ids = set(df_cpc['patent_id'].unique())

# One row per patent with all matching CPC categories pipe-separated
cpc_per_patent = (df_cpc.groupby('patent_id')['cpc_category']
                  .apply(lambda x: '|'.join(sorted(set(x.astype(str)))))
                  .reset_index().rename(columns={'cpc_category': 'cpc_categories'}))

print(f'\nUnique green patents: {len(green_ids):,}')
print(df_cpc.groupby('cpc_category')['patent_id'].nunique().rename('unique_patents'))

Loading CPC file...
  Columns: ['patent_id', 'cpc_sequence', 'cpc_section', 'cpc_class', 'cpc_subclass', 'cpc_group', 'cpc_type']
  patent_id col: "patent_id"  |  cpc col: "cpc_subclass"
  Chunk 1: 500,000 rows scanned | 8,744 green rows kept
  Chunk 2: 1,000,000 rows scanned | 18,116 green rows kept
  Chunk 3: 1,500,000 rows scanned | 27,410 green rows kept
  Chunk 4: 2,000,000 rows scanned | 36,388 green rows kept
  Chunk 5: 2,500,000 rows scanned | 44,730 green rows kept
  Chunk 6: 3,000,000 rows scanned | 52,022 green rows kept
  Chunk 7: 3,500,000 rows scanned | 58,455 green rows kept
  Chunk 8: 4,000,000 rows scanned | 64,384 green rows kept
  Chunk 9: 4,500,000 rows scanned | 70,282 green rows kept
  Chunk 10: 5,000,000 rows scanned | 76,233 green rows kept
  Chunk 11: 5,500,000 rows scanned | 82,158 green rows kept
  Chunk 12: 6,000,000 rows scanned | 88,007 green rows kept
  Chunk 13: 6,500,000 rows scanned | 93,881 green rows kept
  Chunk 14: 7,000,000 rows scanned | 99,319 g

In [ ]:
# Cell 4: Load application file — extract filing dates for green patents
print('Loading application file (filing dates)...')
sample = pd.read_csv(APP_FILE, sep='\t', nrows=3)
cols = list(sample.columns)
print('  Columns:', cols)

pid_col  = next((c for c in cols if 'patent' in c.lower() and 'id' in c.lower()), cols[0])
date_col = next((c for c in cols if 'filing' in c.lower() or 'date' in c.lower()), None)
print(f'  patent_id col: "{pid_col}"  |  date col: "{date_col}"')

usecols = [c for c in [pid_col, date_col] if c]
app_chunks, scanned = [], 0
for i, chunk in enumerate(pd.read_csv(APP_FILE, sep='\t', chunksize=500_000,
                                        usecols=usecols, low_memory=False)):
    chunk[pid_col] = chunk[pid_col].astype(str)
    match = chunk[chunk[pid_col].isin(green_ids)].copy()
    if date_col:
        match['filing_year'] = pd.to_datetime(match[date_col], errors='coerce').dt.year
        match = match[match['filing_year'].between(YEAR_START, YEAR_END)]
    if len(match): app_chunks.append(match[[pid_col, date_col, 'filing_year']] if date_col else match[[pid_col]])
    scanned += len(chunk)
    print(f'  Chunk {i+1}: ~{scanned:,} scanned | {sum(len(c) for c in app_chunks):,} green filing records kept')

df_app = pd.concat(app_chunks, ignore_index=True) if app_chunks else pd.DataFrame()
df_app = df_app.rename(columns={pid_col: 'patent_id', date_col: 'filing_date'})
df_app = df_app.drop_duplicates(subset='patent_id')

# Update green_ids to only those with a valid filing date in range
if not df_app.empty:
    dated_ids = set(df_app['patent_id'].unique())
    print(f'\nGreen patents with filing date in {YEAR_START}-{YEAR_END}: {len(dated_ids):,}')
    print(df_app['filing_year'].value_counts().sort_index())
else:
    dated_ids = set()
    print(f'\nNo green patents found with filing date in {YEAR_START}-{YEAR_END}.')


Loading application file (filing dates)...
  Columns: ['application_id', 'patent_id', 'patent_application_type', 'filing_date', 'series_code', 'rule_47_flag']
  patent_id col: "patent_id"  |  date col: "filing_date"


/tmp/ipykernel_15149/501048112.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  match['filing_year'] = pd.to_datetime(match[date_col], errors='coerce').dt.year


  Chunk 1: ~500,000 scanned | 0 green filing records kept
  Chunk 2: ~1,000,000 scanned | 0 green filing records kept
  Chunk 3: ~1,500,000 scanned | 0 green filing records kept
  Chunk 4: ~2,000,000 scanned | 0 green filing records kept
  Chunk 5: ~2,500,000 scanned | 0 green filing records kept
  Chunk 6: ~3,000,000 scanned | 0 green filing records kept
  Chunk 7: ~3,500,000 scanned | 0 green filing records kept
  Chunk 8: ~4,000,000 scanned | 0 green filing records kept
  Chunk 9: ~4,500,000 scanned | 0 green filing records kept
  Chunk 10: ~5,000,000 scanned | 0 green filing records kept
  Chunk 11: ~5,500,000 scanned | 0 green filing records kept
  Chunk 12: ~6,000,000 scanned | 0 green filing records kept
  Chunk 13: ~6,500,000 scanned | 0 green filing records kept
  Chunk 14: ~7,000,000 scanned | 0 green filing records kept
  Chunk 15: ~7,500,000 scanned | 3,457 green filing records kept
  Chunk 16: ~8,000,000 scanned | 44,033 green filing records kept
  Chunk 17: ~8,500,000 sca

In [ ]:
# Cell 5: Build location_id -> state lookup
# Strategy: use g_location_disambiguated.tsv as primary source (clean, small),
# then fall back to g_location_not_disambiguated.tsv for any IDs not resolved.

# ── Part A: Disambiguated location file (primary) ──────────────────────
print('Loading disambiguated location file (primary)...')
sample = pd.read_csv(LOCATION_FILE, sep='\t', nrows=3)
cols = list(sample.columns)
print('  Columns:', cols)

loc_id_col  = 'location_id'
state_col   = next((c for c in cols if 'state'   in c.lower()), None)
country_col = next((c for c in cols if 'country' in c.lower()), None)
print(f'  location_id: "{loc_id_col}" | state: "{state_col}" | country: "{country_col}"')

df_loc_disambig = pd.read_csv(LOCATION_FILE, sep='\t',
                               usecols=[loc_id_col, state_col, country_col],
                               low_memory=False)
df_loc_disambig[country_col] = df_loc_disambig[country_col].astype(str).str.upper()
df_loc_disambig[state_col]   = df_loc_disambig[state_col].astype(str).str.upper()
df_loc_disambig_us = df_loc_disambig[
    (df_loc_disambig[country_col] == 'US') &
    (df_loc_disambig[state_col].isin(US_STATES))
][[loc_id_col, state_col]].copy()
df_loc_disambig_us.columns = ['location_id', 'state']
df_loc_disambig_us['location_id'] = df_loc_disambig_us['location_id'].astype(str)
df_loc_disambig_us = df_loc_disambig_us.drop_duplicates(subset='location_id')
print(f'  Disambiguated US location IDs: {len(df_loc_disambig_us):,}')

# ── Part B: Raw (not-disambiguated) location file (fallback) ──────────────
RAW_LOCATION_FILE = f'{FOLDER}/g_location_not_disambiguated.tsv'
import os
if os.path.exists(RAW_LOCATION_FILE):
    print('\nLoading raw location file (fallback for unresolved IDs)...')
    raw_sample = pd.read_csv(RAW_LOCATION_FILE, sep='\t', nrows=3)
    raw_cols = list(raw_sample.columns)
    print('  Columns:', raw_cols)

    # rawlocation_id is the join key; also has location_id (disambig pointer),
    # raw_state, raw_country
    raw_id_col      = next((c for c in raw_cols if c.lower() == 'rawlocation_id'), None)
    raw_loc_ptr_col = next((c for c in raw_cols if c.lower() == 'location_id'), None)
    raw_state_col   = next((c for c in raw_cols if 'state'   in c.lower()), None)
    raw_country_col = next((c for c in raw_cols if 'country' in c.lower()), None)
    print(f'  rawlocation_id: "{raw_id_col}" | location_id ptr: "{raw_loc_ptr_col}" | state: "{raw_state_col}"')

    raw_usecols = list(dict.fromkeys([c for c in [raw_id_col, raw_loc_ptr_col, raw_state_col, raw_country_col] if c]))
    raw_loc_chunks = []
    for i, chunk in enumerate(pd.read_csv(RAW_LOCATION_FILE, sep='\t', chunksize=2_000_000,
                                           usecols=raw_usecols, low_memory=False)):
        chunk[raw_country_col] = chunk[raw_country_col].astype(str).str.upper()
        chunk[raw_state_col]   = chunk[raw_state_col].astype(str).str.upper()
        us = chunk[
            (chunk[raw_country_col] == 'US') &
            (chunk[raw_state_col].isin(US_STATES))
        ].copy()
        if len(us): raw_loc_chunks.append(us)
        print(f'  Chunk {i+1}: {(i+1)*2_000_000:,} rows | {sum(len(c) for c in raw_loc_chunks):,} US rows kept')

    df_raw_loc = pd.concat(raw_loc_chunks, ignore_index=True) if raw_loc_chunks else pd.DataFrame()

    # Build two lookup bridges from the raw file:
    # Bridge 1: location_id (UUID) -> state  (covers assignee/inventor using UUID location_ids)
    # Bridge 2: rawlocation_id -> state       (covers files using integer rawlocation_ids)
    bridges = []
    if raw_loc_ptr_col:
        b1 = df_raw_loc[[raw_loc_ptr_col, raw_state_col]].copy()
        b1.columns = ['location_id', 'state']
        b1['location_id'] = b1['location_id'].astype(str)
        b1 = b1[b1['location_id'] != 'NAN'].drop_duplicates(subset='location_id')
        bridges.append(b1)
        print(f'  Raw->UUID bridge: {len(b1):,} unique location_ids')
    if raw_id_col:
        b2 = df_raw_loc[[raw_id_col, raw_state_col]].copy()
        b2.columns = ['location_id', 'state']
        b2['location_id'] = b2['location_id'].astype(str)
        b2 = b2.drop_duplicates(subset='location_id')
        bridges.append(b2)
        print(f'  Raw ID bridge:    {len(b2):,} unique rawlocation_ids')

    # Combine: disambiguated first (most trusted), then fill gaps from raw bridges
    df_loc_us = pd.concat([df_loc_disambig_us] + bridges, ignore_index=True)
    df_loc_us = df_loc_us.drop_duplicates(subset='location_id', keep='first')  # disambig wins
else:
    print('\nRaw location file not found — using disambiguated only.')
    print('To improve assignee state coverage, download g_location_not_disambiguated.tsv')
    df_loc_us = df_loc_disambig_us

us_location_ids = set(df_loc_us['location_id'].unique())
print(f'\nTotal unique US location_ids in combined lookup: {len(us_location_ids):,}')
print(f'States represented: {sorted(df_loc_us["state"].unique())}')


Loading location file...
  Columns: ['location_id', 'disambig_city', 'disambig_state', 'disambig_country', 'latitude', 'longitude', 'county', 'state_fips', 'county_fips']
  location col: "location_id"  |  state col: "disambig_state"  |  country col: "disambig_country"
  Chunk 1: 1,000,000 rows scanned | 28,873 US rows kept

  Unique US location_ids: 28,873
  States represented: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


In [ ]:
!wc -l "{LOCATION_FILE}"

100453 /content/drive/MyDrive/SBLY210/patent_data/g_location_disambiguated.tsv


In [ ]:
# Cell 6: Load assignee file — map location_id to US state
print('Loading assignee file...')
sample = pd.read_csv(ASSIGNEE_FILE, sep='\t', nrows=3)
cols = list(sample.columns)
print('  Columns:', cols)

pid_col = next((c for c in cols if 'patent' in c.lower() and 'id' in c.lower()), 'patent_id')
loc_col = next((c for c in cols if 'location' in c.lower() and 'id' in c.lower()), 'location_id')
print(f'  patent_id col: "{pid_col}" | location_id col: "{loc_col}"')

valid_pids = dated_ids if 'dated_ids' in globals() and dated_ids else green_ids

usecols = list(dict.fromkeys([pid_col, loc_col]))
assignee_chunks, scanned = [], 0
for i, chunk in enumerate(pd.read_csv(ASSIGNEE_FILE, sep='\t', chunksize=500_000,
                                       usecols=usecols, low_memory=False)):
    chunk[pid_col] = chunk[pid_col].astype(str)
    chunk[loc_col] = chunk[loc_col].astype(str)
    match = chunk[chunk[pid_col].isin(valid_pids)].copy()
    if len(match): assignee_chunks.append(match)
    scanned += len(chunk)
    if (i + 1) % 5 == 0:
        print(f'  Chunk {i+1}: ~{scanned:,} scanned | {sum(len(c) for c in assignee_chunks):,} green assignee records')

df_assignee = pd.concat(assignee_chunks, ignore_index=True) if assignee_chunks else pd.DataFrame()
df_assignee = df_assignee.rename(columns={pid_col: 'patent_id', loc_col: 'location_id'})
print(f'\nTotal green assignee records: {len(df_assignee):,}')

# Left join — keeps all assignee records, NaN state for non-US / unresolved
df_final_assignee = df_assignee.merge(df_loc_us, on='location_id', how='left')

resolved   = df_final_assignee['state'].notna().sum()
unresolved = df_final_assignee['state'].isna().sum()
print(f'Mapped to US state:       {resolved:,}  ({resolved/len(df_final_assignee)*100:.1f}%)')
print(f'No US state (foreign/NaN): {unresolved:,}  ({unresolved/len(df_final_assignee)*100:.1f}%)')

# Keep only rows that resolved to a US state
df_final_assignee = df_final_assignee[df_final_assignee['state'].isin(US_STATES)].copy()
print(f'Retained US assignee rows: {len(df_final_assignee):,} | {df_final_assignee["patent_id"].nunique():,} unique patents')
display(df_final_assignee.head())


Loading assignee file...
  Columns: ['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last', 'disambig_assignee_organization', 'assignee_type', 'location_id']
  patent_id col: "patent_id"  |  location_id col: "location_id"
  Chunk 5: ~2,500,000 scanned | 42,935 green assignee records kept
  Chunk 10: ~5,000,000 scanned | 85,509 green assignee records kept
  Chunk 15: ~7,500,000 scanned | 129,251 green assignee records kept

Total green assignee records extracted: 151,202

Merging with location data to assign US states...
Records successfully mapped to a US state: 55,320


,patent_id,location_id,state
0,11872626,15c69712-16c8-11ed-9b5f-1234bde3cd05,CA
1,12255972,c1ee6332-16c7-11ed-9b5f-1234bde3cd05,CA
2,11035250,05db232c-16c8-11ed-9b5f-1234bde3cd05,NY
3,10781429,c324ce6e-16c7-11ed-9b5f-1234bde3cd05,CA
4,10659403,92237ca2-16c8-11ed-9b5f-1234bde3cd05,NY


In [ ]:
# Cell 7: Load inventor file — extract location_id and map to state
print('Loading inventor file...')
usecols = ['patent_id', 'location_id']
inventor_chunks, scanned = [], 0

for i, chunk in enumerate(pd.read_csv(INVENTOR_FILE, sep='\t', chunksize=500_000,
                                      usecols=usecols, low_memory=False)):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    chunk['location_id'] = chunk['location_id'].astype(str)
    match = chunk[chunk['patent_id'].isin(valid_pids)].copy()
    if len(match): inventor_chunks.append(match)
    scanned += len(chunk)
    if (i + 1) % 5 == 0:
        print(f'  Chunk {i+1}: ~{scanned:,} scanned | {sum(len(c) for c in inventor_chunks):,} green inventor records kept')

df_inventor = pd.concat(inventor_chunks, ignore_index=True) if inventor_chunks else pd.DataFrame()

print('\nMerging with location data to assign US states...')
df_final_inventor = df_inventor.merge(df_loc_us, on='location_id', how='inner')
print(f'Inventor records successfully mapped to a US state: {len(df_final_inventor):,}')

Loading inventor file...
  Chunk 5: ~2,500,000 scanned | 49,119 green inventor records kept
  Chunk 10: ~5,000,000 scanned | 98,264 green inventor records kept
  Chunk 15: ~7,500,000 scanned | 147,262 green inventor records kept
  Chunk 20: ~10,000,000 scanned | 196,044 green inventor records kept
  Chunk 25: ~12,500,000 scanned | 244,868 green inventor records kept
  Chunk 30: ~15,000,000 scanned | 293,846 green inventor records kept
  Chunk 35: ~17,500,000 scanned | 343,014 green inventor records kept
  Chunk 40: ~20,000,000 scanned | 392,101 green inventor records kept
  Chunk 45: ~22,500,000 scanned | 440,909 green inventor records kept

Merging with location data to assign US states...
Inventor records successfully mapped to a US state: 176,989


In [ ]:
# Cell 8: Generate green_patents_raw.csv
print('Aggregating states per patent...')
# Collapse multiple states per patent into a pipe-separated string
assignee_states = df_final_assignee.groupby('patent_id')['state'].apply(lambda x: '|'.join(sorted(set(x)))).reset_index(name='assignee_states')
inventor_states = df_final_inventor.groupby('patent_id')['state'].apply(lambda x: '|'.join(sorted(set(x)))).reset_index(name='inventor_states')

print('Building raw dataset...')
# Start with our base of valid green patents
df_raw = cpc_per_patent[cpc_per_patent['patent_id'].isin(valid_pids)].copy()

# Merge dates, assignee states, and inventor states
df_raw = df_raw.merge(df_app[['patent_id', 'filing_date', 'filing_year']], on='patent_id', how='left')
df_raw = df_raw.merge(assignee_states, on='patent_id', how='left')
df_raw = df_raw.merge(inventor_states, on='patent_id', how='left')

# Add placeholders for title and type since they weren't loaded to save memory
df_raw['title'] = 'N/A'
df_raw['type'] = 'N/A'

# Reorder columns to match requested structure
raw_cols = ['patent_id', 'filing_date', 'title', 'type', 'cpc_categories', 'inventor_states', 'assignee_states']
df_raw = df_raw[raw_cols]

out_raw = f'{FOLDER}/green_patents_raw.csv'
df_raw.to_csv(out_raw, index=False)
print(f'\nSaved raw data to: {out_raw}')
display(df_raw.head())

Aggregating states per patent...
Building raw dataset...

Saved raw data to: /content/drive/MyDrive/SBLY210/patent_data/green_patents_raw.csv


,patent_id,filing_date,title,type,cpc_categories,inventor_states,assignee_states
0,10003254,2018-01-19,N/A,N/A,Y02B,NaN,NaN
1,10008723,2018-02-19,N/A,N/A,Y02E,OH,OH
2,10014490,2018-01-11,N/A,N/A,Y02E|Y02P,NaN,NaN
3,10014537,2018-01-19,N/A,N/A,Y02E,MA,MA
4,10014778,2018-01-12,N/A,N/A,Y02B,NaN,NaN


In [ ]:
# Cell 9: Generate green_patents_panel_inventor.csv
print('Building Inventor Panel...')

# Base table: Patents exploded by CPC category, merged with Year and Inventor State
df_inv_base = df_cpc[df_cpc['patent_id'].isin(valid_pids)].merge(df_app[['patent_id', 'filing_year']], on='patent_id', how='inner')
df_inv_base = df_inv_base.merge(df_final_inventor[['patent_id', 'state']], on='patent_id', how='inner')

# Deduplicate in case a patent has multiple inventors in the exact same state for the exact same CPC
df_inv_base = df_inv_base.drop_duplicates(subset=['patent_id', 'state', 'cpc_category'])

# Pivot table: count unique patents per state/year/cpc
inv_cpc_counts = df_inv_base.groupby(['state', 'filing_year', 'cpc_category'])['patent_id'].nunique().reset_index()
inv_panel = inv_cpc_counts.pivot_table(index=['state', 'filing_year'], columns='cpc_category', values='patent_id', fill_value=0).reset_index()

# Calculate total green patents per state-year (ignoring CPC to avoid double-counting patents with multiple CPCs)
totals = df_inv_base.drop_duplicates(subset=['patent_id', 'state']).groupby(['state', 'filing_year'])['patent_id'].nunique().reset_index(name='total_green_patents')

# Merge totals into panel
inv_panel = inv_panel.merge(totals, on=['state', 'filing_year'], how='left')
inv_panel.columns.name = None # Clean up index name

out_inv = f'{FOLDER}/green_patents_panel_inventor.csv'
inv_panel.to_csv(out_inv, index=False)
print(f'Saved inventor panel to: {out_inv}')
display(inv_panel.head())

Building Inventor Panel...
Saved inventor panel to: /content/drive/MyDrive/SBLY210/patent_data/green_patents_panel_inventor.csv


,state,filing_year,Y02B,Y02C,Y02D,Y02E,Y02P,Y02T,Y02W,total_green_patents
0,AK,2018,0.0,0.0,0.0,0.0,1.0,1.0,0.0,2
1,AK,2019,1.0,0.0,0.0,2.0,2.0,1.0,0.0,4
2,AK,2020,0.0,2.0,0.0,4.0,2.0,0.0,0.0,5
3,AK,2021,0.0,1.0,0.0,2.0,1.0,3.0,0.0,5
4,AK,2022,0.0,0.0,0.0,3.0,2.0,1.0,0.0,5


In [ ]:
# Cell 10: Generate green_patents_panel_assignee.csv
print('Building Assignee Panel...')

# Base table: Patents exploded by CPC category, merged with Year and Assignee State
df_asg_base = df_cpc[df_cpc['patent_id'].isin(valid_pids)].merge(df_app[['patent_id', 'filing_year']], on='patent_id', how='inner')
df_asg_base = df_asg_base.merge(df_final_assignee[['patent_id', 'state']], on='patent_id', how='inner')

# Deduplicate in case a patent has multiple assignees in the exact same state for the exact same CPC
df_asg_base = df_asg_base.drop_duplicates(subset=['patent_id', 'state', 'cpc_category'])

# Pivot table: count unique patents per state/year/cpc
asg_cpc_counts = df_asg_base.groupby(['state', 'filing_year', 'cpc_category'])['patent_id'].nunique().reset_index()
asg_panel = asg_cpc_counts.pivot_table(index=['state', 'filing_year'], columns='cpc_category', values='patent_id', fill_value=0).reset_index()

# Calculate total green patents per state-year
totals_asg = df_asg_base.drop_duplicates(subset=['patent_id', 'state']).groupby(['state', 'filing_year'])['patent_id'].nunique().reset_index(name='total_green_patents')

# Merge totals into panel
asg_panel = asg_panel.merge(totals_asg, on=['state', 'filing_year'], how='left')
asg_panel.columns.name = None

out_asg = f'{FOLDER}/green_patents_panel_assignee.csv'
asg_panel.to_csv(out_asg, index=False)
print(f'Saved assignee panel to: {out_asg}')
display(asg_panel.head())

Building Assignee Panel...
Saved assignee panel to: /content/drive/MyDrive/SBLY210/patent_data/green_patents_panel_assignee.csv


,state,filing_year,Y02B,Y02C,Y02D,Y02E,Y02P,Y02T,Y02W,total_green_patents
0,AK,2018,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
1,AK,2019,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1
2,AK,2020,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1
3,AK,2021,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2
4,AK,2022,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
